# AltFreezing Deepfake Detection — Colab Pipeline
**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Download `model.pth` from https://rec.ustc.edu.cn/share/e87360b0-7b2e-11ef-aeef-a9fd0832d537 (password: `altf`)
3. Upload `model.pth` to Google Drive at `My Drive/altfreezing/model.pth`
4. Run cells top to bottom

In [ ]:
# Cell 1 — GPU check
import torch
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}  |  torch {torch.__version__}")

In [ ]:
# Cell 2 — Clone AltFreezing + install deps
import os
from pathlib import Path

ALTFREEZING = Path('/content/AltFreezing')
if not ALTFREEZING.exists():
    !git clone -q https://github.com/ZhendongWang6/AltFreezing.git {ALTFREEZING}
    print('Cloned')
else:
    print('Already cloned')

%pip install -q \
    opencv_transforms termcolor tensorboardX contexttimer imgaug \
    prettytable simplejson fvcore ffmpeg-python tabulate \
    lmdb PyTurboJPEG albumentations einops filterpy scikit-learn
print('Deps installed')

In [ ]:
# Cell 3 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 4 — CONFIG  ← only cell you need to edit
CKPT_PATH = '/content/drive/MyDrive/altfreezing/model.pth'
INPUT_DIR  = '/content/drive/MyDrive/deepfake videos/archive/DFD_manipulated_sequences/DFD_manipulated_sequences'
OUTPUT_DIR = '/content/results'

assert Path(CKPT_PATH).exists(), (
    f"model.pth not found at {CKPT_PATH}\n"
    "Download from https://rec.ustc.edu.cn/share/e87360b0-7b2e-11ef-aeef-a9fd0832d537 (password: altf)"
)
videos = sorted(Path(INPUT_DIR).rglob('*.mp4'))
assert videos, f'No .mp4 files under {INPUT_DIR}'
print(f'Checkpoint : {CKPT_PATH}')
print(f'Videos     : {len(videos)}')

In [ ]:
# Cell 5 — chdir + sys.path  (MUST run before importing AltFreezing)
import sys
os.chdir(str(ALTFREEZING))                       # face weights download to ./auxillary/ (CWD-relative)
if str(ALTFREEZING) not in sys.path:
    sys.path.insert(0, str(ALTFREEZING))
print('CWD:', os.getcwd())

In [ ]:
%%writefile /content/detector.py
"""
detector.py — AltFreezing model loading + per-video inference.
Caller must os.chdir(ALTFREEZING_DIR) and sys.path.insert before importing.
"""
import numpy as np
import torch
import torch.nn.functional as F

from config import config as cfg
from test_tools.common import detect_all
from test_tools.ct.operations import find_longest, multiple_tracking
from test_tools.faster_crop_align_xray import FasterCropAlignXRay
from test_tools.utils import get_crop_box
from utils.plugin_loader import PluginLoader

THRESHOLD  = 0.04
_MAX_FRAME = 400
_MEAN = torch.tensor([0.485*255, 0.456*255, 0.406*255]).view(1,3,1,1,1)
_STD  = torch.tensor([0.229*255, 0.224*255, 0.225*255]).view(1,3,1,1,1)


def load_model(ckpt_path):
    """Returns (classifier, crop_align_func)."""
    cfg.init_with_yaml()
    cfg.update_with_yaml('i3d_ori.yaml')
    cfg.freeze()
    classifier = PluginLoader.get_classifier(cfg.classifier_type)()
    classifier.cuda()
    classifier.eval()
    classifier.load(ckpt_path)
    return classifier, FasterCropAlignXRay(cfg.imsize)


def infer_video(video_path, classifier, crop_align_func):
    """Returns mean sigmoid score. Score >= THRESHOLD means FAKE. Returns 0.5 if no face found."""
    detect_res, all_lm68, frames = detect_all(str(video_path), return_frames=True, max_size=_MAX_FRAME)
    if not frames:
        return 0.5
    shape = frames[0].shape[:2]

    merged = []
    for faces, lm68s in zip(detect_res, all_lm68):
        merged.append([(box, lm5, lm68, score) for (box, lm5, score), lm68 in zip(faces, lm68s)])
    detect_res = merged

    tracks = multiple_tracking(detect_res)
    tuples = [(0, len(detect_res))] * len(tracks)
    if not tracks:
        tuples, tracks = find_longest(detect_res)
    if not tracks:
        return 0.5

    data_storage = {}
    super_clips = []
    for track_i, ((start, end), track) in enumerate(zip(tuples, tracks)):
        super_clips.append(len(track))
        for j, (face, frame_idx) in enumerate(zip(track, range(start, end))):
            box, lm5, lm68 = face[:3]
            big_box  = get_crop_box(shape, box, scale=0.5)
            top_left = big_box[:2][None, :]
            info = ((box.reshape(2,2)-top_left).reshape(-1), lm5-top_left, lm68-top_left, big_box)
            x1, y1, x2, y2 = big_box
            data_storage[f'{track_i}_{j}_img'] = frames[frame_idx][y1:y2, x1:x2]
            data_storage[f'{track_i}_{j}_ldm'] = info
            data_storage[f'{track_i}_{j}_idx'] = frame_idx

    clip_size  = cfg.clip_size
    pad_length = clip_size - 1
    clips = []
    for sc_idx, sc_size in enumerate(super_clips):
        inner = list(range(sc_size))
        if sc_size < clip_size:
            post = inner[1:-1][::-1] + inner
            post = (post * (pad_length // len(post) + 1))[:pad_length]
            pre  = inner + inner[1:-1][::-1]
            pre  = (pre  * (pad_length // len(pre)  + 1))[-pad_length:]
            inner = pre + inner + post
        sc_size = len(inner)
        for i in range(sc_size):
            if i + clip_size <= sc_size:
                clips.append([(sc_idx, inner[i+k]) for k in range(clip_size)])
    if not clips:
        return 0.5

    mean, std = _MEAN.cuda(), _STD.cuda()
    preds = []
    for clip in clips:
        imgs = [data_storage[f'{i}_{j}_img'] for i, j in clip]
        lmks = [data_storage[f'{i}_{j}_ldm'] for i, j in clip]
        _, imgs_align = crop_align_func(lmks, imgs)
        t = torch.as_tensor(imgs_align, dtype=torch.float32).cuda().permute(3,0,1,2).unsqueeze(0)
        t = t.sub(mean).div(std)
        with torch.no_grad():
            out = classifier(t)
        preds.append(float(F.sigmoid(out['final_output'])))
    return float(np.mean(preds))

In [ ]:
# Cell 7 — Import detector + load model
# First import triggers face detector weight downloads (~100 MB, ~1 min first run)
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

import importlib, detector
importlib.reload(detector)          # safe to re-run cell
from detector import load_model, infer_video, THRESHOLD

print('Loading AltFreezing model...')
classifier, crop_align_func = load_model(CKPT_PATH)
print(f'Ready  |  threshold={THRESHOLD}')

In [ ]:
# Cell 8 — Run inference (streams results to CSV)
import csv
from datetime import datetime
from tqdm.notebook import tqdm

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
csv_path = Path(OUTPUT_DIR) / f"results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['video', 'score', 'prediction'])

    for video in tqdm(videos, desc='Inferring'):
        try:
            score = infer_video(video, classifier, crop_align_func)
        except Exception as e:
            tqdm.write(f'  ERROR {video.name}: {e}')
            score = -1.0
        prediction = 'FAKE' if score >= THRESHOLD else 'real'
        writer.writerow([video.name, f'{score:.4f}', prediction])
        f.flush()
        tqdm.write(f'  {video.name:<55} {prediction:<6}  ({score:.4f})')

print(f'Done → {csv_path}')


In [ ]:
# Cell 9 — Download CSV
from google.colab import files
files.download(str(csv_path))

# AltFreezing Deepfake Detection — Colab Pipeline
**Model:** AltFreezing (CVPR 2023) · I3D spatiotemporal network · 32-frame clips  
**Threshold:** 0.04 (score ≥ 0.04 → FAKE)  
**Before running:** put `model.pth` from RecDrive/BaiduDrive (password: `altf`) in your Google Drive at `My Drive/altfreezing/model.pth`

In [ ]:
# ── Cell 1: GPU check ─────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}  |  torch {torch.__version__}")

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
# Colab already has: torch, torchvision, numpy, opencv, pandas, scipy, pillow
%pip install -q \
    opencv_transforms termcolor tensorboardX contexttimer imgaug \
    prettytable simplejson fvcore ffmpeg-python tabulate \
    lmdb PyTurboJPEG albumentations einops filterpy scikit-learn

In [ ]:
# ── Cell 3: Clone AltFreezing ─────────────────────────────────────────────────
import os
from pathlib import Path

ALTFREEZING_DIR = Path('/content/AltFreezing')

if not ALTFREEZING_DIR.exists():
    !git clone https://github.com/ZhendongWang6/AltFreezing.git {ALTFREEZING_DIR}
else:
    print('AltFreezing already cloned')

# MUST chdir so face-detector weights download to ./auxillary/ (CWD-relative)
os.chdir(str(ALTFREEZING_DIR))
import sys
if str(ALTFREEZING_DIR) not in sys.path:
    sys.path.insert(0, str(ALTFREEZING_DIR))
print('CWD:', os.getcwd())

In [ ]:
# ── Cell 4: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 5: CONFIG — edit paths if different ──────────────────────────────────

# Where model.pth lives in your Drive
CKPT_PATH  = '/content/drive/MyDrive/altfreezing/model.pth'

# Input folder — all .mp4 files inside will be processed  
INPUT_DIR  = '/content/drive/MyDrive/deepfake videos/archive/DFD_manipulated_sequences/DFD_manipulated_sequences'

# Where CSV results go
OUTPUT_DIR = '/content/results'

# Score threshold from demo.py — don't change
THRESHOLD  = 0.04

# ── sanity checks ────────────────────────────────────────────────────────────
assert Path(CKPT_PATH).exists(), (
    f"model.pth not found at {CKPT_PATH}\n"
    "Download from RecDrive (password: altf): "
    "https://rec.ustc.edu.cn/share/e87360b0-7b2e-11ef-aeef-a9fd0832d537\n"
    "Then upload to Google Drive at the path above."
)
videos = sorted(Path(INPUT_DIR).rglob('*.mp4'))
assert videos, f'No .mp4 files found under {INPUT_DIR}'
print(f'Checkpoint : {CKPT_PATH}')
print(f'Input      : {INPUT_DIR}')
print(f'Videos     : {len(videos)}')

In [ ]:
# ── Cell 6: Load model ────────────────────────────────────────────────────────
# NOTE: importing test_tools.common triggers automatic weight downloads
# for FaceDetector + LandmarkPredictor (~100 MB) — takes ~1 min first run.
import torch
import torch.nn.functional as F

from config import config as cfg
from test_tools.common import detect_all
from test_tools.ct.operations import find_longest, multiple_tracking
from test_tools.faster_crop_align_xray import FasterCropAlignXRay
from test_tools.utils import get_crop_box
from utils.plugin_loader import PluginLoader

cfg.init_with_yaml()
cfg.update_with_yaml('i3d_ori.yaml')
cfg.freeze()

classifier = PluginLoader.get_classifier(cfg.classifier_type)()
classifier.cuda()
classifier.eval()
classifier.load(CKPT_PATH)

crop_align_func = FasterCropAlignXRay(cfg.imsize)
print(f'Model loaded  imsize={cfg.imsize}  clip_size={cfg.clip_size}  threshold={THRESHOLD}')

In [ ]:
# ── Cell 7: Inference helpers ─────────────────────────────────────────────────
import numpy as np

_MEAN = torch.tensor([0.485*255, 0.456*255, 0.406*255]).cuda().view(1,3,1,1,1)
_STD  = torch.tensor([0.229*255, 0.224*255, 0.225*255]).cuda().view(1,3,1,1,1)
_MAX_FRAME = 400

def infer_video(video_path):
    detect_res, all_lm68, frames = detect_all(
        str(video_path), return_frames=True, max_size=_MAX_FRAME
    )
    if not frames:
        return 0.5

    shape = frames[0].shape[:2]

    # merge 5-pt + 68-pt landmarks
    merged = []
    for faces, lm68s in zip(detect_res, all_lm68):
        merged.append([
            (box, lm5, lm68, score)
            for (box, lm5, score), lm68 in zip(faces, lm68s)
        ])
    detect_res = merged

    tracks = multiple_tracking(detect_res)
    tuples = [(0, len(detect_res))] * len(tracks)
    if not tracks:
        tuples, tracks = find_longest(detect_res)
    if not tracks:
        return 0.5

    data_storage = {}
    super_clips   = []

    for track_i, ((start, end), track) in enumerate(zip(tuples, tracks)):
        super_clips.append(len(track))
        for j, (face, frame_idx) in enumerate(zip(track, range(start, end))):
            box, lm5, lm68 = face[:3]
            big_box  = get_crop_box(shape, box, scale=0.5)
            top_left = big_box[:2][None, :]
            info = (
                (box.reshape(2,2) - top_left).reshape(-1),
                lm5  - top_left,
                lm68 - top_left,
                big_box,
            )
            x1, y1, x2, y2 = big_box
            data_storage[f'{track_i}_{j}_img'] = frames[frame_idx][y1:y2, x1:x2]
            data_storage[f'{track_i}_{j}_ldm'] = info

    clip_size  = cfg.clip_size
    pad_length = clip_size - 1
    clips = []

    for sc_idx, sc_size in enumerate(super_clips):
        inner = list(range(sc_size))
        if sc_size < clip_size:
            post = inner[1:-1][::-1] + inner
            post = (post * (pad_length // len(post) + 1))[:pad_length]
            pre  = inner + inner[1:-1][::-1]
            pre  = (pre  * (pad_length // len(pre)  + 1))[-pad_length:]
            inner = pre + inner + post
        sc_size = len(inner)
        for i in range(sc_size):
            if i + clip_size <= sc_size:
                clips.append([(sc_idx, inner[i+k]) for k in range(clip_size)])

    if not clips:
        return 0.5

    preds = []
    for clip in clips:
        imgs = [data_storage[f'{i}_{j}_img'] for i, j in clip]
        lmks = [data_storage[f'{i}_{j}_ldm'] for i, j in clip]
        _, imgs_align = crop_align_func(lmks, imgs)
        t = torch.as_tensor(imgs_align, dtype=torch.float32).cuda().permute(3,0,1,2).unsqueeze(0)
        t = t.sub(_MEAN).div(_STD)
        with torch.no_grad():
            out = classifier(t)
        preds.append(float(F.sigmoid(out['final_output'])))

    return float(np.mean(preds))

print('Helper functions ready')

In [ ]:
# ── Cell 8: Run inference (streams results to CSV) ────────────────────────────
import csv
from datetime import datetime
from tqdm.notebook import tqdm

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
csv_path = Path(OUTPUT_DIR) / f"results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['video', 'score', 'prediction'])

    for video in tqdm(videos, desc='Inferring'):
        try:
            score = infer_video(video)
        except Exception as e:
            tqdm.write(f'  ERROR {video.name}: {e}')
            score = -1.0

        prediction = 'FAKE' if score >= THRESHOLD else 'real'
        writer.writerow([video.name, f'{score:.4f}', prediction])
        f.flush()
        tqdm.write(f'  {video.name:<55} {prediction:<6}  ({score:.4f})')

print(f'\nDone. Results → {csv_path}')


In [ ]:
# ── Cell 9: Download CSV ──────────────────────────────────────────────────────
from google.colab import files
files.download(str(csv_path))

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — no GPU detected')

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'efficientnet_pytorch', 'retinaface_pytorch', 'opencv-python',
    'tqdm', 'scikit-learn', 'pandas', 'gdown', 'pretrainedmodels', 'albumentations',
])

In [ ]:
import sys, subprocess
from pathlib import Path

SBI_DIR = Path('/content/SelfBlendedImages')
if not SBI_DIR.exists():
    subprocess.check_call(['git', 'clone', '--depth=1',
        'https://github.com/mapooon/SelfBlendedImages.git', str(SBI_DIR)])
sys.path.insert(0, str(SBI_DIR / 'src'))
print('SBI src ready')

In [ ]:
import gdown
from pathlib import Path

WEIGHTS_DIR = Path('/content/weights')
WEIGHTS_DIR.mkdir(exist_ok=True)

weights_files = {
    'FFraw': ('12sLyqBp0VFwdpA-oZLdIOkOTkz_ZnIhV', WEIGHTS_DIR / 'FFraw.tar'),
    'FFc23': ('1X0-NYT8KPursLZZdxduRQju6E52hauV0',  WEIGHTS_DIR / 'FFc23.tar'),
}
for name, (fid, dest) in weights_files.items():
    if not dest.exists():
        print(f'Downloading {name}...')
        gdown.download(id=fid, output=str(dest), quiet=False)
    else:
        print(f'{name} already cached.')

weights_paths = [str(v[1]) for v in weights_files.values()]
print('Weights ready:', weights_paths)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

INPUT_DIR = Path('/content/drive/MyDrive/deepfake videos/archive/DFD_manipulated_sequences/DFD_manipulated_sequences')

video_list = sorted(INPUT_DIR.glob('*.mp4'))
assert video_list, f'No .mp4 files found in {INPUT_DIR}'
print(f'{len(video_list)} video(s) found')

In [ ]:
import numpy as np, torch, csv
from datetime import datetime
from tqdm.notebook import tqdm
from retinaface.pre_trained_models import get_model
from model import Detector
from inference.preprocess import extract_frames

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_FRAMES = 32

def load_model(path):
    m = Detector().to(DEVICE)
    m.load_state_dict(torch.load(path, map_location=DEVICE)['model'])
    m.eval()
    return m

def score_faces(faces, idx_list, model):
    if not faces:
        return 0.5
    with torch.no_grad():
        preds = model(torch.from_numpy(np.stack(faces)).to(DEVICE).float() / 255).softmax(1)[:, 1]
    buckets = {}
    for idx, p in zip(idx_list, preds.tolist()):
        buckets.setdefault(idx, []).append(p)
    return float(np.mean([max(v) for v in buckets.values()]))

print('Loading models...')
models = [load_model(p) for p in weights_paths]
face_detector = get_model('resnet50_2020-07-20', max_size=2048, device=DEVICE)
face_detector.eval()
print(f'Ready on {DEVICE}')

In [ ]:
OUTPUT_DIR = Path('/content/results')
OUTPUT_DIR.mkdir(exist_ok=True)
out_file = OUTPUT_DIR / f"results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

with open(out_file, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['video', 'score', 'prediction'])
    for video in tqdm(video_list, desc='Inferring'):
        faces, idx_list = extract_frames(str(video), N_FRAMES, face_detector)
        score = max(score_faces(faces, idx_list, m) for m in models)
        prediction = 'FAKE' if score >= 0.5 else 'real'
        w.writerow([video.name, f'{score:.4f}', prediction])
        f.flush()
        tqdm.write(f'  {video.name:<55} {prediction:<6}  ({score:.4f})')

print(f'Done. Results -> {out_file}')

In [ ]:
from google.colab import files
from pathlib import Path

# pick up out_file from the inference cell, or fall back to the latest CSV in /content/results/
if 'out_file' not in dir():
    results = sorted(Path('/content/results').glob('results_*.csv'))
    assert results, "No results CSV found — run the inference cell first"
    out_file = results[-1]

files.download(str(out_file))
print('Downloaded:', out_file)
